In [23]:
from google.colab import drive
import os
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs")

# If you are using a run folder like outputs/run_YYYYMMDD_HHMMSS
RUN_ID = "run_20260209_185402"   # <-- your current run id
RUN_DIR = os.path.join(OUT_DIR, RUN_ID)

TEXT_OUT = os.path.join(RUN_DIR, "text_extracted")
REPORT_OUT = os.path.join(RUN_DIR, "reports")

os.makedirs(TEXT_OUT, exist_ok=True)
os.makedirs(REPORT_OUT, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("TEXT_OUT:", TEXT_OUT, "| exists:", os.path.exists(TEXT_OUT))
print("REPORT_OUT:", REPORT_OUT, "| exists:", os.path.exists(REPORT_OUT))


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402
TEXT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted | exists: True
REPORT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/reports | exists: True


In [24]:
import os, re, json
import pandas as pd
from datetime import datetime

# ✅ Paste the SAME RUN_ID created in Notebook 1
RUN_ID = "20260209_185402"   # e.g. "20260210_234501"

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"
RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_20260209_185402")

TEXT_OUT = os.path.join(RUN_DIR, "text_extracted")
REPORT_OUT = os.path.join(RUN_DIR, "reports")

CHUNK_OUT = os.path.join(RUN_DIR, "chunks")
os.makedirs(CHUNK_OUT, exist_ok=True)

print("✅ Chunking notebook ready")
print("RUN_DIR:", RUN_DIR)
print("TEXT_OUT:", TEXT_OUT)
print("CHUNK_OUT:", CHUNK_OUT)


✅ Chunking notebook ready
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402
TEXT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted
CHUNK_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks


In [25]:
def list_text_files(text_dir: str):
    if not os.path.exists(text_dir):
        raise FileNotFoundError(f"TEXT_OUT not found: {text_dir}")

    files = [os.path.join(text_dir, f) for f in os.listdir(text_dir) if f.lower().endswith(".txt")]
    files.sort()
    return files

text_files = list_text_files(TEXT_OUT)

print("✅ Extracted text files found:", len(text_files))
for f in text_files[:20]:
    print(" -", os.path.basename(f))
if len(text_files) > 20:
    print("... (showing first 20)")


✅ Extracted text files found: 13
 - Farming_Schemes_pdf.txt
 - farmerbook_pdf.txt
 - web_ALL.txt
 - web_Agricultural_Prices.txt
 - web_Crop_Area_Statistics.txt
 - web_Crop_Forecasts.txt
 - web_Crop_Production.txt
 - web_Fisheries_Statistics.txt
 - web_Forestry_Statistics.txt
 - web_Introduction.txt
 - web_Irrigation_Statistics.txt
 - web_Land_Use.txt
 - web_Production_of_Horticultural_Crops.txt


In [26]:
def show_text_preview(title: str, text: str, n_chars: int = 800):
    print("\n" + "="*90)
    print("PREVIEW:", title)
    print("="*90)
    print(text[:n_chars])
    print("\n[Preview chars shown:", min(len(text), n_chars), "of", len(text), "]")

def infer_source_type(filename: str):
    # convention from Notebook 1:
    # pdf -> "*_pdf.txt"
    # web -> "web_*.txt" or "web_ALL.txt"
    fn = filename.lower()
    if fn.endswith("_pdf.txt"):
        return "pdf"
    if fn.startswith("web_"):
        return "web"
    return "unknown"

def infer_source_name(filename: str):
    # For PDFs: "farmerbook_pdf.txt" -> "farmerbook"
    # For web: "web_Introduction.txt" -> "Introduction"
    fn = os.path.basename(filename)
    if fn.lower().endswith("_pdf.txt"):
        return fn[:-8]  # remove "_pdf.txt"
    if fn.lower().startswith("web_") and fn.lower().endswith(".txt"):
        return fn[4:-4]  # remove "web_" and ".txt"
    return fn.replace(".txt","")


In [27]:
def chunk_text_words(text: str, chunk_size_words: int = 220, overlap_words: int = 50):
    """
    Simple, robust chunking for RAG:
    - Splits by words (stable across PDFs/web)
    - Uses overlap to keep context continuity
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    n = len(words)

    while start < n:
        end = min(start + chunk_size_words, n)
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words).strip()
        if chunk:
            chunks.append(chunk)

        if end == n:
            break
        start = max(0, end - overlap_words)

    return chunks


In [29]:
import re
def safe_filename(name: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_")
    return name[:120] if len(name) > 120 else name


In [30]:

# ✅ Chunking parameters (tune later)
CHUNK_SIZE_WORDS = 220
OVERLAP_WORDS = 50

rows = []
global_chunk_counter = 0

print("\n" + "="*90)
print("STEP: CHUNKING ALL TEXT FILES")
print("="*90)

for path in text_files:
    fname = os.path.basename(path)
    source_type = infer_source_type(fname)
    source_name = infer_source_name(fname)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    total_chars = len(text)
    total_words = len(text.split())

    # Create chunks
    chunks = chunk_text_words(text, CHUNK_SIZE_WORDS, OVERLAP_WORDS)

    print("\n" + "-"*90)
    print("File:", fname)
    print("source_type:", source_type, "| source_name:", source_name)
    print("Total words:", total_words, "| Total chars:", total_chars)
    print("Chunks created:", len(chunks))

    # Add rows
    for i, chunk in enumerate(chunks, start=1):
        global_chunk_counter += 1
        chunk_id = f"{source_type}_{safe_filename(source_name)}_{i:04d}_{global_chunk_counter:06d}"

        rows.append({
            "chunk_id": chunk_id,
            "source_type": source_type,
            "source_name": source_name,
            "file_name": fname,
            "file_path": path,
            "chunk_index_in_file": i,
            "chunk_size_words": CHUNK_SIZE_WORDS,
            "overlap_words": OVERLAP_WORDS,
            "chunk_words": len(chunk.split()),
            "chunk_chars": len(chunk),
            "chunk_text": chunk
        })

# Final dataframe
chunks_df = pd.DataFrame(rows)

print("\n✅ TOTAL CHUNKS:", len(chunks_df))
print("✅ Total sources:", chunks_df["file_name"].nunique())
display(chunks_df.head(10))



STEP: CHUNKING ALL TEXT FILES

------------------------------------------------------------------------------------------
File: Farming_Schemes_pdf.txt
source_type: pdf | source_name: Farming_Schemes
Total words: 94954 | Total chars: 638828
Chunks created: 559

------------------------------------------------------------------------------------------
File: farmerbook_pdf.txt
source_type: pdf | source_name: farmerbook
Total words: 43759 | Total chars: 278092
Chunks created: 258

------------------------------------------------------------------------------------------
File: web_ALL.txt
source_type: web | source_name: ALL
Total words: 12388 | Total chars: 80317
Chunks created: 73

------------------------------------------------------------------------------------------
File: web_Agricultural_Prices.txt
source_type: web | source_name: Agricultural_Prices
Total words: 949 | Total chars: 6190
Chunks created: 6

------------------------------------------------------------------------------

,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...
5,pdf_Farming_Schemes_0006_000006,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,6,220,50,220,1478,and Small Solar Power Plants Programme 201 17....
6,pdf_Farming_Schemes_0007_000007,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,7,220,50,220,1381,Education among Scheduled Tribe (ST) Girls in ...
7,pdf_Farming_Schemes_0008_000008,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,8,220,50,220,1307,Support Price Scheme Type of scheme : Central ...
8,pdf_Farming_Schemes_0009_000009,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,9,220,50,220,1340,support prices are a guarantee price for their...
9,pdf_Farming_Schemes_0010_000010,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,10,220,50,220,1527,bumper production years. The minimum support p...


In [31]:
print("\n" + "="*90)
print("STEP: CHUNK STATS (PRESENTATION READY)")
print("="*90)

# 1) Summary by source file
by_file = chunks_df.groupby(["source_type","file_name"], as_index=False).agg(
    chunks=("chunk_id","count"),
    total_words=("chunk_words","sum"),
    avg_chunk_words=("chunk_words","mean"),
    avg_chunk_chars=("chunk_chars","mean")
).sort_values("chunks", ascending=False)

print("✅ Summary by file (top 10):")
display(by_file.head(10))

# 2) Summary by source_type
by_type = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name","nunique"),
    chunks=("chunk_id","count"),
    avg_chunk_words=("chunk_words","mean"),
    avg_chunk_chars=("chunk_chars","mean")
)
print("✅ Summary by source_type:")
display(by_type)

# 3) Show a few example chunks (first chunk per file)
examples = chunks_df.sort_values(["file_name","chunk_index_in_file"]).groupby("file_name").head(1)
print("✅ Example: first chunk from each file (preview 5):")
display(examples[["file_name","chunk_id","chunk_words","chunk_text"]].head(5))



STEP: CHUNK STATS (PRESENTATION READY)
✅ Summary by file (top 10):


,source_type,file_name,chunks,total_words,avg_chunk_words,avg_chunk_chars
0,pdf,Farming_Schemes_pdf.txt,559,122854,219.774597,1447.345259
1,pdf,farmerbook_pdf.txt,258,56609,219.414729,1384.825581
2,web,web_ALL.txt,73,15988,219.013699,1416.657534
4,web,web_Crop_Area_Statistics.txt,17,3584,210.823529,1306.058824
6,web,web_Crop_Production.txt,10,2081,208.100000,1307.900000
10,web,web_Irrigation_Statistics.txt,8,1673,209.125000,1359.250000
3,web,web_Agricultural_Prices.txt,6,1199,199.833333,1302.333333
7,web,web_Fisheries_Statistics.txt,6,1260,210.000000,1362.166667
5,web,web_Crop_Forecasts.txt,6,1236,206.000000,1392.500000
8,web,web_Forestry_Statistics.txt,6,1190,198.333333,1294.666667


✅ Summary by source_type:


,source_type,files,chunks,avg_chunk_words,avg_chunk_chars
0,pdf,2,817,219.660955,1427.602203
1,web,11,149,212.590604,1374.234899


✅ Example: first chunk from each file (preview 5):


,file_name,chunk_id,chunk_words,chunk_text
0,Farming_Schemes_pdf.txt,pdf_Farming_Schemes_0001_000001,220,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
559,farmerbook_pdf.txt,pdf_farmerbook_0001_000560,220,A holistic perspective of scientific agricultu...
817,web_ALL.txt,web_ALL_0001_000818,220,SECTION: Introduction URL: https://nsc.mospi.g...
890,web_Agricultural_Prices.txt,web_Agricultural_Prices_0001_000891,220,SECTION: Agricultural Prices URL: https://nsc....
896,web_Crop_Area_Statistics.txt,web_Crop_Area_Statistics_0001_000897,220,SECTION: Crop Area Statistics URL: https://nsc...


In [32]:
def preview_chunk(chunk_id: str, n_chars: int = 900):
    row = chunks_df[chunks_df["chunk_id"] == chunk_id]
    if row.empty:
        print("❌ chunk_id not found:", chunk_id)
        return
    r = row.iloc[0]
    title = f"{r['chunk_id']} | {r['source_type']} | {r['file_name']} | chunk {r['chunk_index_in_file']}"
    show_text_preview(title, r["chunk_text"], n_chars=n_chars)

# ✅ Try previewing first chunk
preview_chunk(chunks_df.iloc[0]["chunk_id"])



PREVIEW: pdf_Farming_Schemes_0001_000001 | pdf | Farming_Schemes_pdf.txt | chunk 1
COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND PROGRAMMES RELEVANT TO THE NORTH EASTERN STATES Compiled by: NERCORMP NORTH EASTERN REGION COMMUNITY RESOURCE MANAGEMENT PROJECT SHILLONG - 793 003 November 2020 1 CONTENTS Name of Schemes Page No. 1. Ministry of Agriculture & Farmers Welfare 1.1 Minimum Support Price Scheme 10 1.2 Marketing research and information network (under Integrated Scheme for Agricultural Marketing -ISAM) 11 1.3 Central Herd Registration Scheme 12 1.4 National Food Security Mission 13 1.5 Rashtriya Krishi Vikas Yojana Remunerative Approaches for Agriculture and Allied sector Rejuvenation (RKVY -RAFTAAR) 15 1.6 National Mission for Sustainable Agriculture (NMSA) 16 1.7 National Innovations on Climate Resilient Agriculture (NICRA) 20 1.8 Integrated Scheme for Agricultural Marketing (ISAM) 22 1.9 Mission for Integrated Development of Horticulture (MIDH) 24 1.10 Sub-Mis

[Preview chars

In [33]:
chunks_csv = os.path.join(CHUNK_OUT, "chunks.csv")
chunks_parquet = os.path.join(CHUNK_OUT, "chunks.parquet")
preview_csv = os.path.join(CHUNK_OUT, "chunks_preview_top10.csv")

# Full dataset
chunks_df.to_csv(chunks_csv, index=False)
chunks_df.to_parquet(chunks_parquet, index=False)

# Small preview for sharing
chunks_df.head(10).to_csv(preview_csv, index=False)

print("✅ Saved full chunks CSV:", chunks_csv)
print("✅ Saved full chunks Parquet:", chunks_parquet)
print("✅ Saved preview CSV:", preview_csv)


✅ Saved full chunks CSV: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks.csv
✅ Saved full chunks Parquet: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks.parquet
✅ Saved preview CSV: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks_preview_top10.csv


In [34]:
log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "input_text_dir": TEXT_OUT,
    "output_chunk_dir": CHUNK_OUT,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "overlap_words": OVERLAP_WORDS,
    "num_files": int(chunks_df["file_name"].nunique()),
    "num_chunks": int(len(chunks_df)),
    "source_type_counts": chunks_df["source_type"].value_counts().to_dict()
}

log_path = os.path.join(CHUNK_OUT, "chunking_run_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("✅ Chunking log saved:", log_path)
print(json.dumps(log, indent=2))


✅ Chunking log saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunking_run_log.json
{
  "run_id": "20260209_185402",
  "created_at": "2026-02-09T19:42:59.747956",
  "input_text_dir": "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted",
  "output_chunk_dir": "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks",
  "chunk_size_words": 220,
  "overlap_words": 50,
  "num_files": 13,
  "num_chunks": 966,
  "source_type_counts": {
    "pdf": 817,
    "web": 149
  }
}


In [35]:
# ============================================================
# NOTEBOOK 2: CHUNKING (FINAL CLEANED + PRESENTATION READY)
# Output:
#  - chunks/chunks.csv
#  - chunks/chunks.parquet
#  - chunks/chunks_preview_top10.csv
#  - chunks/chunking_run_log.json
#  - chunks/chunk_stats_by_file.csv
#  - chunks/chunk_stats_by_type.csv
#  - chunks/chunk_examples_first_per_file.csv
# ============================================================

# -----------------------------
# Cell 1 — Mount Drive (safe)
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, re, json
import pandas as pd
from datetime import datetime

# -----------------------------
# Cell 2 — Set PROJECT + RUN
# -----------------------------
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"

# ✅ IMPORTANT: use the same RUN_ID created in Dataset Prep notebook
RUN_ID = "20260209_185402"  # <-- paste yours here (WITHOUT "run_")

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")
TEXT_OUT = os.path.join(RUN_DIR, "text_extracted")
REPORT_OUT = os.path.join(RUN_DIR, "reports")

CHUNK_OUT = os.path.join(RUN_DIR, "chunks")
os.makedirs(CHUNK_OUT, exist_ok=True)

print("✅ Chunking notebook ready")
print("RUN_DIR:", RUN_DIR, "| exists:", os.path.exists(RUN_DIR))
print("TEXT_OUT:", TEXT_OUT, "| exists:", os.path.exists(TEXT_OUT))
print("CHUNK_OUT:", CHUNK_OUT, "| exists:", os.path.exists(CHUNK_OUT))


# -----------------------------
# Cell 3 — Utilities
# -----------------------------
def list_text_files(text_dir: str):
    if not os.path.exists(text_dir):
        raise FileNotFoundError(f"TEXT_OUT not found: {text_dir}")

    files = [os.path.join(text_dir, f) for f in os.listdir(text_dir) if f.lower().endswith(".txt")]
    files.sort()
    return files

def show_text_preview(title: str, text: str, n_chars: int = 800):
    print("\n" + "="*90)
    print("PREVIEW:", title)
    print("="*90)
    print(text[:n_chars])
    print("\n[Preview chars shown:", min(len(text), n_chars), "of", len(text), "]")

def infer_source_type(filename: str):
    fn = filename.lower()
    if fn.endswith("_pdf.txt"):
        return "pdf"
    if fn.startswith("web_"):
        return "web"
    return "unknown"

def infer_source_name(filename: str):
    fn = os.path.basename(filename)
    if fn.lower().endswith("_pdf.txt"):
        return fn[:-8]  # remove "_pdf.txt"
    if fn.lower().startswith("web_") and fn.lower().endswith(".txt"):
        return fn[4:-4]  # remove "web_" and ".txt"
    return fn.replace(".txt","")

def safe_filename(name: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_")
    return name[:120] if len(name) > 120 else name

def chunk_text_words(text: str, chunk_size_words: int = 220, overlap_words: int = 50):
    """
    RAG-friendly chunking:
    - word-based chunking (stable across PDF/Web)
    - overlap keeps context continuity
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    n = len(words)

    while start < n:
        end = min(start + chunk_size_words, n)
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words).strip()
        if chunk:
            chunks.append(chunk)

        if end == n:
            break
        start = max(0, end - overlap_words)

    return chunks

def remove_common_nav_noise(text: str) -> str:
    """
    Light cleanup for web pages that include repeated navbar text.
    Keep it conservative: do not over-delete content.
    """
    if not text:
        return ""
    patterns = [
        r"\bSKIP TO MAIN CONTENT\b",
        r"\bSCREEN READER ACCESS\b",
        r"\bAccessibility Controls\b",
        r"\bHome\b\s*\bAbout us\b\s*\bDocuments\b\s*\bRTI\b\s*\bGallery\b\s*\bContact Us\b",
    ]
    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, " ", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


# -----------------------------
# Cell 4 — Verify inputs
# -----------------------------
text_files = list_text_files(TEXT_OUT)

print("✅ Extracted text files found:", len(text_files))
for f in text_files[:20]:
    print(" -", os.path.basename(f))
if len(text_files) > 20:
    print("... (showing first 20)")


# -----------------------------
# Cell 5 — Chunking Parameters
# -----------------------------
CHUNK_SIZE_WORDS = 220
OVERLAP_WORDS = 50

print("\nChunking params:")
print("CHUNK_SIZE_WORDS:", CHUNK_SIZE_WORDS)
print("OVERLAP_WORDS:", OVERLAP_WORDS)


# -----------------------------
# Cell 6 — Chunk ALL files (with dedupe fix)
# -----------------------------
rows = []
global_chunk_counter = 0

print("\n" + "="*90)
print("STEP: CHUNKING ALL TEXT FILES")
print("="*90)

for path in text_files:
    fname = os.path.basename(path)

    # ✅ IMPORTANT: skip combined web_ALL to avoid duplicated content
    if fname.lower() == "web_all.txt":
        print("\nSkipping duplicate combined web file:", fname)
        continue

    source_type = infer_source_type(fname)
    source_name = infer_source_name(fname)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # Optional: clean web navbar junk
    if source_type == "web":
        text = remove_common_nav_noise(text)

    total_chars = len(text)
    total_words = len(text.split())

    chunks = chunk_text_words(text, CHUNK_SIZE_WORDS, OVERLAP_WORDS)

    print("\n" + "-"*90)
    print("File:", fname)
    print("source_type:", source_type, "| source_name:", source_name)
    print("Total words:", total_words, "| Total chars:", total_chars)
    print("Chunks created:", len(chunks))

    for i, chunk in enumerate(chunks, start=1):
        global_chunk_counter += 1
        chunk_id = f"{source_type}_{safe_filename(source_name)}_{i:04d}_{global_chunk_counter:06d}"

        rows.append({
            "chunk_id": chunk_id,
            "source_type": source_type,
            "source_name": source_name,
            "file_name": fname,
            "file_path": path,
            "chunk_index_in_file": i,
            "chunk_size_words": CHUNK_SIZE_WORDS,
            "overlap_words": OVERLAP_WORDS,
            "chunk_words": len(chunk.split()),
            "chunk_chars": len(chunk),
            "chunk_preview": chunk[:220],
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(rows)

print("\n✅ TOTAL CHUNKS:", len(chunks_df))
print("✅ Total sources (post-skip):", chunks_df["file_name"].nunique())
display(chunks_df.head(10))


# -----------------------------
# Cell 7 — Presentation-ready stats
# -----------------------------
print("\n" + "="*90)
print("STEP: CHUNK STATS (PRESENTATION READY)")
print("="*90)

by_file = chunks_df.groupby(["source_type","file_name"], as_index=False).agg(
    chunks=("chunk_id","count"),
    total_words=("chunk_words","sum"),
    avg_chunk_words=("chunk_words","mean"),
    avg_chunk_chars=("chunk_chars","mean")
).sort_values("chunks", ascending=False)

print("✅ Summary by file (top 10):")
display(by_file.head(10))

by_type = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name","nunique"),
    chunks=("chunk_id","count"),
    avg_chunk_words=("chunk_words","mean"),
    avg_chunk_chars=("chunk_chars","mean")
)

print("✅ Summary by source_type:")
display(by_type)

examples = chunks_df.sort_values(["file_name","chunk_index_in_file"]).groupby("file_name").head(1)
print("✅ Example: first chunk from each file (preview 5):")
display(examples[["file_name","chunk_id","chunk_words","chunk_text"]].head(5))


# -----------------------------
# Cell 8 — Quick chunk preview helper
# -----------------------------
def preview_chunk(chunk_id: str, n_chars: int = 900):
    row = chunks_df[chunks_df["chunk_id"] == chunk_id]
    if row.empty:
        print("❌ chunk_id not found:", chunk_id)
        return
    r = row.iloc[0]
    title = f"{r['chunk_id']} | {r['source_type']} | {r['file_name']} | chunk {r['chunk_index_in_file']}"
    show_text_preview(title, r["chunk_text"], n_chars=n_chars)

# Preview first chunk (safe)
preview_chunk(chunks_df.iloc[0]["chunk_id"])


# -----------------------------
# Cell 9 — Save outputs
# -----------------------------
chunks_csv = os.path.join(CHUNK_OUT, "chunks.csv")
chunks_parquet = os.path.join(CHUNK_OUT, "chunks.parquet")
preview_csv = os.path.join(CHUNK_OUT, "chunks_preview_top10.csv")

chunks_df.to_csv(chunks_csv, index=False)
chunks_df.to_parquet(chunks_parquet, index=False)
chunks_df.head(10).to_csv(preview_csv, index=False)

print("✅ Saved full chunks CSV:", chunks_csv)
print("✅ Saved full chunks Parquet:", chunks_parquet)
print("✅ Saved preview CSV:", preview_csv)

# Save stats for presentation
by_file_path = os.path.join(CHUNK_OUT, "chunk_stats_by_file.csv")
by_type_path = os.path.join(CHUNK_OUT, "chunk_stats_by_type.csv")
examples_path = os.path.join(CHUNK_OUT, "chunk_examples_first_per_file.csv")

by_file.to_csv(by_file_path, index=False)
by_type.to_csv(by_type_path, index=False)
examples[["file_name","chunk_id","chunk_words","chunk_text"]].to_csv(examples_path, index=False)

print("✅ Saved presentation stats:", by_file_path)
print("✅ Saved presentation stats:", by_type_path)
print("✅ Saved examples:", examples_path)


# -----------------------------
# Cell 10 — Save run log
# -----------------------------
log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "input_text_dir": TEXT_OUT,
    "output_chunk_dir": CHUNK_OUT,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "overlap_words": OVERLAP_WORDS,
    "num_files_input_txt": int(len(list_text_files(TEXT_OUT))),
    "num_files_chunked": int(chunks_df["file_name"].nunique()),
    "num_chunks": int(len(chunks_df)),
    "source_type_counts": chunks_df["source_type"].value_counts().to_dict(),
    "skipped_files": ["web_ALL.txt"]
}

log_path = os.path.join(CHUNK_OUT, "chunking_run_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("✅ Chunking log saved:", log_path)
print(json.dumps(log, indent=2))


Mounted at /content/drive
✅ Chunking notebook ready
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402 | exists: True
TEXT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted | exists: True
CHUNK_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks | exists: True
✅ Extracted text files found: 13
 - Farming_Schemes_pdf.txt
 - farmerbook_pdf.txt
 - web_ALL.txt
 - web_Agricultural_Prices.txt
 - web_Crop_Area_Statistics.txt
 - web_Crop_Forecasts.txt
 - web_Crop_Production.txt
 - web_Fisheries_Statistics.txt
 - web_Forestry_Statistics.txt
 - web_Introduction.txt
 - web_Irrigation_Statistics.txt
 - web_Land_Use.txt
 - web_Production_of_Horticultural_Crops.txt

Chunking params:
CHUNK_SIZE_WORDS: 220
OVERLAP_WORDS: 50

STEP: CHUNKING ALL TEXT FILES

------------------------------------------------------------------------------------

,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_preview,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...,National TB Control Programme (RNTCP) 158 12. ...
5,pdf_Farming_Schemes_0006_000006,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,6,220,50,220,1478,and Small Solar Power Plants Programme 201 17....,and Small Solar Power Plants Programme 201 17....
6,pdf_Farming_Schemes_0007_000007,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,7,220,50,220,1381,Education among Scheduled Tribe (ST) Girls in ...,Education among Scheduled Tribe (ST) Girls in ...
7,pdf_Farming_Schemes_0008_000008,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,8,220,50,220,1307,Support Price Scheme Type of scheme : Central ...,Support Price Scheme Type of scheme : Central ...
8,pdf_Farming_Schemes_0009_000009,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,9,220,50,220,1340,support prices are a guarantee price for their...,support prices are a guarantee price for their...
9,pdf_Farming_Schemes_0010_000010,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,10,220,50,220,1527,bumper production years. The minimum support p...,bumper production years. The minimum support p...



STEP: CHUNK STATS (PRESENTATION READY)
✅ Summary by file (top 10):


,source_type,file_name,chunks,total_words,avg_chunk_words,avg_chunk_chars
0,pdf,Farming_Schemes_pdf.txt,559,122854,219.774597,1447.345259
1,pdf,farmerbook_pdf.txt,258,56609,219.414729,1384.825581
3,web,web_Crop_Area_Statistics.txt,16,3509,219.312500,1364.125000
5,web,web_Crop_Production.txt,10,2056,205.600000,1286.300000
9,web,web_Irrigation_Statistics.txt,8,1648,206.000000,1346.500000
2,web,web_Agricultural_Prices.txt,6,1174,195.666667,1254.833333
6,web,web_Fisheries_Statistics.txt,6,1235,205.833333,1338.500000
4,web,web_Crop_Forecasts.txt,6,1211,201.833333,1356.333333
7,web,web_Forestry_Statistics.txt,6,1165,194.166667,1267.666667
8,web,web_Introduction.txt,6,1295,215.833333,1477.666667


✅ Summary by source_type:


,source_type,files,chunks,avg_chunk_words,avg_chunk_chars
0,pdf,2,817,219.660955,1427.602203
1,web,10,74,207.270270,1337.945946


✅ Example: first chunk from each file (preview 5):


,file_name,chunk_id,chunk_words,chunk_text
0,Farming_Schemes_pdf.txt,pdf_Farming_Schemes_0001_000001,220,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
559,farmerbook_pdf.txt,pdf_farmerbook_0001_000560,220,A holistic perspective of scientific agricultu...
817,web_Agricultural_Prices.txt,web_Agricultural_Prices_0001_000818,220,SECTION: Agricultural Prices URL: https://nsc....
823,web_Crop_Area_Statistics.txt,web_Crop_Area_Statistics_0001_000824,220,SECTION: Crop Area Statistics URL: https://nsc...
839,web_Crop_Forecasts.txt,web_Crop_Forecasts_0001_000840,220,SECTION: Crop Forecasts URL: https://nsc.mospi...



PREVIEW: pdf_Farming_Schemes_0001_000001 | pdf | Farming_Schemes_pdf.txt | chunk 1
COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND PROGRAMMES RELEVANT TO THE NORTH EASTERN STATES Compiled by: NERCORMP NORTH EASTERN REGION COMMUNITY RESOURCE MANAGEMENT PROJECT SHILLONG - 793 003 November 2020 1 CONTENTS Name of Schemes Page No. 1. Ministry of Agriculture & Farmers Welfare 1.1 Minimum Support Price Scheme 10 1.2 Marketing research and information network (under Integrated Scheme for Agricultural Marketing -ISAM) 11 1.3 Central Herd Registration Scheme 12 1.4 National Food Security Mission 13 1.5 Rashtriya Krishi Vikas Yojana Remunerative Approaches for Agriculture and Allied sector Rejuvenation (RKVY -RAFTAAR) 15 1.6 National Mission for Sustainable Agriculture (NMSA) 16 1.7 National Innovations on Climate Resilient Agriculture (NICRA) 20 1.8 Integrated Scheme for Agricultural Marketing (ISAM) 22 1.9 Mission for Integrated Development of Horticulture (MIDH) 24 1.10 Sub-Mis

[Preview chars

In [1]:
# ============================================================
# NOTEBOOK 2: CHUNKING (FINAL CLEANED + PRESENTATION READY)
# Output:
#  - chunks/chunks.csv
#  - chunks/chunks.parquet
#  - chunks/chunks_preview_top10.csv
#  - chunks/chunking_run_log.json
#  - chunks/chunk_stats_by_file.csv
#  - chunks/chunk_stats_by_type.csv
#  - chunks/chunk_examples_first_per_file.csv
# ============================================================

# -----------------------------
# Cell 1 — Mount Drive (safe)
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, re, json
import pandas as pd
from datetime import datetime

# -----------------------------
# Cell 2 — Set PROJECT + RUN
# -----------------------------
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"

# ✅ IMPORTANT: use the same RUN_ID created in Dataset Prep notebook (WITHOUT "run_")
RUN_ID = "20260209_185402"

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")
TEXT_OUT = os.path.join(RUN_DIR, "text_extracted")
REPORT_OUT = os.path.join(RUN_DIR, "reports")

CHUNK_OUT = os.path.join(RUN_DIR, "chunks")
os.makedirs(CHUNK_OUT, exist_ok=True)

print("✅ Chunking notebook ready")
print("RUN_DIR:", RUN_DIR, "| exists:", os.path.exists(RUN_DIR))
print("TEXT_OUT:", TEXT_OUT, "| exists:", os.path.exists(TEXT_OUT))
print("CHUNK_OUT:", CHUNK_OUT, "| exists:", os.path.exists(CHUNK_OUT))

# -----------------------------
# Cell 3 — Utilities
# -----------------------------
def list_text_files(text_dir: str):
    if not os.path.exists(text_dir):
        raise FileNotFoundError(f"TEXT_OUT not found: {text_dir}")
    files = [os.path.join(text_dir, f) for f in os.listdir(text_dir) if f.lower().endswith(".txt")]
    files.sort()
    return files

def show_text_preview(title: str, text: str, n_chars: int = 800):
    print("\n" + "="*90)
    print("PREVIEW:", title)
    print("="*90)
    print(text[:n_chars])
    print("\n[Preview chars shown:", min(len(text), n_chars), "of", len(text), "]")

def infer_source_type(filename: str):
    fn = filename.lower()
    if fn.endswith("_pdf.txt"):
        return "pdf"
    if fn.startswith("web_"):
        return "web"
    return "unknown"

def infer_source_name(filename: str):
    fn = os.path.basename(filename)
    if fn.lower().endswith("_pdf.txt"):
        return fn[:-8]  # remove "_pdf.txt"
    if fn.lower().startswith("web_") and fn.lower().endswith(".txt"):
        return fn[4:-4]  # remove "web_" and ".txt"
    return fn.replace(".txt", "")

def safe_filename(name: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_")
    return name[:120] if len(name) > 120 else name

def chunk_text_words(text: str, chunk_size_words: int = 220, overlap_words: int = 50):
    """
    RAG-friendly chunking:
    - word-based chunking (stable across PDF/Web)
    - overlap keeps context continuity
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    n = len(words)

    while start < n:
        end = min(start + chunk_size_words, n)
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words).strip()
        if chunk:
            chunks.append(chunk)

        if end == n:
            break
        start = max(0, end - overlap_words)

    return chunks

def remove_common_nav_noise(text: str) -> str:
    """
    Light cleanup for web pages that include repeated navbar text.
    Keep it conservative: do not over-delete content.
    """
    if not text:
        return ""
    patterns = [
        r"\bSKIP TO MAIN CONTENT\b",
        r"\bSCREEN READER ACCESS\b",
        r"\bAccessibility Controls\b",
        r"\bHome\b\s*\bAbout us\b\s*\bDocuments\b\s*\bRTI\b\s*\bGallery\b\s*\bContact Us\b",
    ]
    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, " ", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()

def read_text_file(path: str) -> str:
    """
    Robust reader:
    - tries utf-8 first
    - falls back to latin-1 (rare PDFs/web dumps)
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except UnicodeDecodeError:
        with open(path, "r", encoding="latin-1") as f:
            return f.read()

def normalize_whitespace(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# -----------------------------
# Cell 4 — Verify inputs + preview
# -----------------------------
text_files = list_text_files(TEXT_OUT)

print("✅ Extracted text files found:", len(text_files))
for f in text_files[:20]:
    print(" -", os.path.basename(f))
if len(text_files) > 20:
    print("... (showing first 20)")

# Optional: quick preview of 1 file (helps presentation)
sample_path = text_files[0] if text_files else None
if sample_path:
    sample_text = normalize_whitespace(read_text_file(sample_path))
    show_text_preview(f"RAW TEXT SAMPLE: {os.path.basename(sample_path)}", sample_text, n_chars=700)

# -----------------------------
# Cell 5 — Chunking Parameters
# -----------------------------
CHUNK_SIZE_WORDS = 220
OVERLAP_WORDS = 50

print("\nChunking params:")
print("CHUNK_SIZE_WORDS:", CHUNK_SIZE_WORDS)
print("OVERLAP_WORDS:", OVERLAP_WORDS)

# -----------------------------
# Cell 6 — Chunk ALL files (dedupe + quality checks)
# -----------------------------
rows = []
global_chunk_counter = 0

skipped_files = []
input_files_scanned = 0
empty_text_files = []
files_with_zero_chunks = []

print("\n" + "="*90)
print("STEP: CHUNKING ALL TEXT FILES")
print("="*90)

for path in text_files:
    input_files_scanned += 1
    fname = os.path.basename(path)

    # ✅ Skip combined web_ALL to avoid duplicated content
    if fname.lower() == "web_all.txt":
        skipped_files.append(fname)
        print("\nSkipping duplicate combined web file:", fname)
        continue

    source_type = infer_source_type(fname)
    source_name = infer_source_name(fname)

    text = normalize_whitespace(read_text_file(path))

    # Optional: clean web navbar junk
    if source_type == "web":
        text = remove_common_nav_noise(text)
        text = normalize_whitespace(text)

    if not text:
        empty_text_files.append(fname)
        print("\n⚠️ Empty text file:", fname, "(skipping)")
        continue

    total_chars = len(text)
    total_words = len(text.split())

    chunks = chunk_text_words(text, CHUNK_SIZE_WORDS, OVERLAP_WORDS)

    if len(chunks) == 0:
        files_with_zero_chunks.append(fname)

    print("\n" + "-"*90)
    print("File:", fname)
    print("source_type:", source_type, "| source_name:", source_name)
    print("Total words:", total_words, "| Total chars:", total_chars)
    print("Chunks created:", len(chunks))

    for i, chunk in enumerate(chunks, start=1):
        global_chunk_counter += 1
        chunk_id = f"{source_type}_{safe_filename(source_name)}_{i:04d}_{global_chunk_counter:06d}"

        rows.append({
            "chunk_id": chunk_id,
            "source_type": source_type,
            "source_name": source_name,
            "file_name": fname,
            "file_path": path,
            "chunk_index_in_file": i,
            "chunk_size_words": CHUNK_SIZE_WORDS,
            "overlap_words": OVERLAP_WORDS,
            "chunk_words": len(chunk.split()),
            "chunk_chars": len(chunk),
            "chunk_preview": chunk[:220],
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(rows)

# Quality checks (useful to explain in viva/presentation)
null_chunk_text = int(chunks_df["chunk_text"].isna().sum()) if len(chunks_df) else 0
empty_chunk_text = int((chunks_df["chunk_text"].astype(str).str.strip() == "").sum()) if len(chunks_df) else 0

print("\n✅ TOTAL CHUNKS:", len(chunks_df))
print("✅ Total sources (post-skip):", chunks_df["file_name"].nunique() if len(chunks_df) else 0)
print("✅ Null chunk_text rows:", null_chunk_text)
print("✅ Empty chunk_text rows:", empty_chunk_text)
if empty_text_files:
    print("⚠️ Empty input text files:", empty_text_files)
if files_with_zero_chunks:
    print("⚠️ Files with zero chunks:", files_with_zero_chunks)
if skipped_files:
    print("ℹ️ Skipped files:", skipped_files)

display(chunks_df.head(10))

# -----------------------------
# Cell 7 — Presentation-ready stats + exportable tables
# -----------------------------
print("\n" + "="*90)
print("STEP: CHUNK STATS (PRESENTATION READY)")
print("="*90)

by_file = chunks_df.groupby(["source_type", "file_name"], as_index=False).agg(
    chunks=("chunk_id", "count"),
    total_words=("chunk_words", "sum"),
    avg_chunk_words=("chunk_words", "mean"),
    avg_chunk_chars=("chunk_chars", "mean"),
).sort_values("chunks", ascending=False)

by_type = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name", "nunique"),
    chunks=("chunk_id", "count"),
    avg_chunk_words=("chunk_words", "mean"),
    avg_chunk_chars=("chunk_chars", "mean"),
)

examples = (
    chunks_df.sort_values(["file_name", "chunk_index_in_file"])
    .groupby("file_name")
    .head(1)
)

print("✅ Summary by file (top 10):")
display(by_file.head(10))

print("✅ Summary by source_type:")
display(by_type)

print("✅ Example: first chunk from each file (preview 5):")
display(examples[["file_name", "chunk_id", "chunk_words", "chunk_text"]].head(5))

# -----------------------------
# Cell 8 — Quick chunk preview helper
# -----------------------------
def preview_chunk(chunk_id: str, n_chars: int = 900):
    row = chunks_df[chunks_df["chunk_id"] == chunk_id]
    if row.empty:
        print("❌ chunk_id not found:", chunk_id)
        return
    r = row.iloc[0]
    title = f"{r['chunk_id']} | {r['source_type']} | {r['file_name']} | chunk {r['chunk_index_in_file']}"
    show_text_preview(title, r["chunk_text"], n_chars=n_chars)

# Preview first chunk (safe)
if len(chunks_df):
    preview_chunk(chunks_df.iloc[0]["chunk_id"])

# -----------------------------
# Cell 9 — Save outputs (CSV + Parquet + preview + stats)
# -----------------------------
chunks_csv = os.path.join(CHUNK_OUT, "chunks.csv")
chunks_parquet = os.path.join(CHUNK_OUT, "chunks.parquet")
preview_csv = os.path.join(CHUNK_OUT, "chunks_preview_top10.csv")

chunks_df.to_csv(chunks_csv, index=False)
chunks_df.to_parquet(chunks_parquet, index=False)
chunks_df.head(10).to_csv(preview_csv, index=False)

print("✅ Saved full chunks CSV:", chunks_csv)
print("✅ Saved full chunks Parquet:", chunks_parquet)
print("✅ Saved preview CSV:", preview_csv)

# Save stats for presentation
by_file_path = os.path.join(CHUNK_OUT, "chunk_stats_by_file.csv")
by_type_path = os.path.join(CHUNK_OUT, "chunk_stats_by_type.csv")
examples_path = os.path.join(CHUNK_OUT, "chunk_examples_first_per_file.csv")

by_file.to_csv(by_file_path, index=False)
by_type.to_csv(by_type_path, index=False)
examples[["file_name", "chunk_id", "chunk_words", "chunk_text"]].to_csv(examples_path, index=False)

print("✅ Saved presentation stats:", by_file_path)
print("✅ Saved presentation stats:", by_type_path)
print("✅ Saved examples:", examples_path)

# -----------------------------
# Cell 10 — Save run log (more complete)
# -----------------------------
log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "project_root": PROJECT_ROOT,
    "run_dir": RUN_DIR,
    "input_text_dir": TEXT_OUT,
    "output_chunk_dir": CHUNK_OUT,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "overlap_words": OVERLAP_WORDS,
    "num_files_input_txt": int(len(text_files)),
    "num_files_chunked": int(chunks_df["file_name"].nunique()) if len(chunks_df) else 0,
    "num_chunks": int(len(chunks_df)),
    "source_type_counts": chunks_df["source_type"].value_counts().to_dict() if len(chunks_df) else {},
    "skipped_files": skipped_files,
    "empty_input_files": empty_text_files,
    "files_with_zero_chunks": files_with_zero_chunks,
    "null_chunk_text_rows": null_chunk_text,
    "empty_chunk_text_rows": empty_chunk_text,
    "saved_files": {
        "chunks_csv": chunks_csv,
        "chunks_parquet": chunks_parquet,
        "preview_csv": preview_csv,
        "chunk_stats_by_file": by_file_path,
        "chunk_stats_by_type": by_type_path,
        "chunk_examples_first_per_file": examples_path,
    }
}

log_path = os.path.join(CHUNK_OUT, "chunking_run_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("✅ Chunking log saved:", log_path)
print(json.dumps(log, indent=2))


Mounted at /content/drive
✅ Chunking notebook ready
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402 | exists: True
TEXT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted | exists: True
CHUNK_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks | exists: True
✅ Extracted text files found: 13
 - Farming_Schemes_pdf.txt
 - farmerbook_pdf.txt
 - web_ALL.txt
 - web_Agricultural_Prices.txt
 - web_Crop_Area_Statistics.txt
 - web_Crop_Forecasts.txt
 - web_Crop_Production.txt
 - web_Fisheries_Statistics.txt
 - web_Forestry_Statistics.txt
 - web_Introduction.txt
 - web_Irrigation_Statistics.txt
 - web_Land_Use.txt
 - web_Production_of_Horticultural_Crops.txt

PREVIEW: RAW TEXT SAMPLE: Farming_Schemes_pdf.txt
COMPENDIUM 
OF 
GOVERNMENT OF INDIA SCHEMES AND 
PROGRAMMES RELEVANT TO THE 
NORTH EASTERN STATES 

 

 

 

 

Compiled by

,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_preview,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...,National TB Control Programme (RNTCP) 158 12. ...
5,pdf_Farming_Schemes_0006_000006,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,6,220,50,220,1478,and Small Solar Power Plants Programme 201 17....,and Small Solar Power Plants Programme 201 17....
6,pdf_Farming_Schemes_0007_000007,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,7,220,50,220,1381,Education among Scheduled Tribe (ST) Girls in ...,Education among Scheduled Tribe (ST) Girls in ...
7,pdf_Farming_Schemes_0008_000008,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,8,220,50,220,1307,Support Price Scheme Type of scheme : Central ...,Support Price Scheme Type of scheme : Central ...
8,pdf_Farming_Schemes_0009_000009,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,9,220,50,220,1340,support prices are a guarantee price for their...,support prices are a guarantee price for their...
9,pdf_Farming_Schemes_0010_000010,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,10,220,50,220,1527,bumper production years. The minimum support p...,bumper production years. The minimum support p...



STEP: CHUNK STATS (PRESENTATION READY)
✅ Summary by file (top 10):


,source_type,file_name,chunks,total_words,avg_chunk_words,avg_chunk_chars
0,pdf,Farming_Schemes_pdf.txt,559,122854,219.774597,1447.345259
1,pdf,farmerbook_pdf.txt,258,56609,219.414729,1384.825581
3,web,web_Crop_Area_Statistics.txt,16,3509,219.312500,1364.125000
5,web,web_Crop_Production.txt,10,2056,205.600000,1286.300000
9,web,web_Irrigation_Statistics.txt,8,1648,206.000000,1346.500000
2,web,web_Agricultural_Prices.txt,6,1174,195.666667,1254.833333
6,web,web_Fisheries_Statistics.txt,6,1235,205.833333,1338.500000
4,web,web_Crop_Forecasts.txt,6,1211,201.833333,1356.333333
7,web,web_Forestry_Statistics.txt,6,1165,194.166667,1267.666667
8,web,web_Introduction.txt,6,1295,215.833333,1477.666667


✅ Summary by source_type:


,source_type,files,chunks,avg_chunk_words,avg_chunk_chars
0,pdf,2,817,219.660955,1427.602203
1,web,10,74,207.270270,1337.945946


✅ Example: first chunk from each file (preview 5):


,file_name,chunk_id,chunk_words,chunk_text
0,Farming_Schemes_pdf.txt,pdf_Farming_Schemes_0001_000001,220,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
559,farmerbook_pdf.txt,pdf_farmerbook_0001_000560,220,A holistic perspective of scientific agricultu...
817,web_Agricultural_Prices.txt,web_Agricultural_Prices_0001_000818,220,SECTION: Agricultural Prices URL: https://nsc....
823,web_Crop_Area_Statistics.txt,web_Crop_Area_Statistics_0001_000824,220,SECTION: Crop Area Statistics URL: https://nsc...
839,web_Crop_Forecasts.txt,web_Crop_Forecasts_0001_000840,220,SECTION: Crop Forecasts URL: https://nsc.mospi...



PREVIEW: pdf_Farming_Schemes_0001_000001 | pdf | Farming_Schemes_pdf.txt | chunk 1
COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND PROGRAMMES RELEVANT TO THE NORTH EASTERN STATES Compiled by: NERCORMP NORTH EASTERN REGION COMMUNITY RESOURCE MANAGEMENT PROJECT SHILLONG - 793 003 November 2020 1 CONTENTS Name of Schemes Page No. 1. Ministry of Agriculture & Farmers Welfare 1.1 Minimum Support Price Scheme 10 1.2 Marketing research and information network (under Integrated Scheme for Agricultural Marketing -ISAM) 11 1.3 Central Herd Registration Scheme 12 1.4 National Food Security Mission 13 1.5 Rashtriya Krishi Vikas Yojana Remunerative Approaches for Agriculture and Allied sector Rejuvenation (RKVY -RAFTAAR) 15 1.6 National Mission for Sustainable Agriculture (NMSA) 16 1.7 National Innovations on Climate Resilient Agriculture (NICRA) 20 1.8 Integrated Scheme for Agricultural Marketing (ISAM) 22 1.9 Mission for Integrated Development of Horticulture (MIDH) 24 1.10 Sub-Mis

[Preview chars